[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_05/1_algorithms.ipynb)

# Day 5: Algorithms (easy)

**How to work:** Name the pattern before you code. Stuck 15 min: open Hint 1. Stuck 25 min: Hint 2, then the solution. Say the time and space complexity out loud. Re-type the solution from memory if you needed it.

**Today (50 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | Ways up the staircase | easy | 15 |
| Q2 | Best stretch of k days | easy | 15 |
| Q3 | Mirror the tree | easy | 10 |
| Q4 | Squeeze out repeats in a sorted list (bonus) | easy | 10 |

**Handbook:** [Algorithms](https://mahsa-agz.github.io/ds-handbook/#algo), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Setup

Run this cell once before the questions.

In [ ]:
import copy

def check(fn, cases, key=None):
    """Runs fn on each (args, expected) case and prints PASS or FAIL. key: normalise outputs (e.g. sorted)."""
    ok = 0
    for args, want in cases:
        args = args if isinstance(args, tuple) else (args,)
        try:
            got = fn(*copy.deepcopy(args))
        except Exception as e:
            got = "error: " + repr(e)
        good = (key(got) == key(want)) if (key and not isinstance(got, str)) else got == want
        ok += good
        shown = ", ".join(repr(a) for a in args)
        print(("PASS " if good else "FAIL ") + f"{fn.__name__}({shown}) -> {got!r}" + ("" if good else f"   expected {want!r}"))
    print(f"{ok}/{len(cases)} passed")

from collections import deque

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def build_tree(vals):
    """Level-order list with None for gaps (LeetCode style) -> root. [1, 2, 3, None, 4] means 2 has a right child 4."""
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = deque([root]), 1
    while queue and i < len(vals):
        node = queue.popleft()
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_values(root):
    """Root -> level-order list with None for gaps (trailing None removed)."""
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if not isinstance(node, TreeNode):
            out.append(None)
            continue
        out.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

The setup cell gives you `TreeNode`, `build_tree([4, 2, 7])` (level order, `None` = no child) and
`tree_values(root)`.

---
### Q1. Ways up the staircase

*easy, about 15 min*

A staircase has `n` steps. Each move you climb either 1 or 2 steps. In how many different orders can you
reach the top? `n` can be up to 45, so trying every sequence one by one is too slow.

Examples:
- `n = 2` -> `2` (1+1, 2)
- `n = 3` -> `3` (1+1+1, 1+2, 2+1)
- `n = 5` -> `8`

*Source: [LeetCode 70](https://leetcode.com/problems/climbing-stairs/)*

In [ ]:
def climb_stairs(n):
    pass

# tests
check(climb_stairs, [
    (1, 1),
    (2, 2),
    (3, 3),
    (5, 8),
    (10, 89),
    (45, 1836311903),
])

<details><summary><b>Hint 1</b></summary>

Signal: the count for `n` is built from the counts for smaller `n` (the last move was 1 or 2 steps). Pattern: **1-D dynamic programming**.

</details>

<details><summary><b>Hint 2</b></summary>

1. State: `ways(i)` = number of ways to reach step `i`.
2. Transition: `ways(i) = ways(i-1) + ways(i-2)`; base cases `ways(1) = 1`, `ways(2) = 2`.
3. You only need the last two values: keep two variables and roll them forward.

</details>

<details><summary><b>Solution</b></summary>

```python
def climb_stairs(n):
    a, b = 1, 1               # ways to reach step 0 and step 1
    for _ in range(n - 1):
        a, b = b, a + b
    return b
```

**Why:** Every route to step n ends with a 1-step from n-1 or a 2-step from n-2, and those two groups do not overlap, so the counts add (it is the Fibonacci sequence). Plain recursion recomputes the same sub-answers again and again (O(2^n)); storing them makes it linear.

**Complexity:** O(n) time, O(1) space.

**Common mistakes:** Plain recursion without memo (times out at n = 45). Wrong base case (`ways(0)` = 1, not 0). Off-by-one in the loop count.

**Learn more:** [algo-dp-1d](https://mahsa-agz.github.io/ds-handbook/#algo-dp-1d)

</details>

---
### Q2. Best stretch of k days

*easy, about 15 min*

`nums` holds daily values and `k` is a window length. Among all blocks of exactly `k` consecutive
values, return the largest average (a float). Aim for O(n), not O(n * k).

Examples:
- `nums = [1, 12, -5, -6, 50, 3], k = 4` -> `12.75` (block `[12, -5, -6, 50]`, sum 51)
- `nums = [0, 4, 0, 3, 2], k = 1` -> `4.0`
- `nums = [-1, -2, -3], k = 2` -> `-1.5`

*Source: [LeetCode 643](https://leetcode.com/problems/maximum-average-subarray-i/)*

In [ ]:
def find_max_average(nums, k):
    pass

# tests
check(find_max_average, [
    (([1, 12, -5, -6, 50, 3], 4), 12.75),
    (([5], 1), 5.0),
    (([0, 4, 0, 3, 2], 1), 4.0),
    (([-1, -2, -3], 2), -1.5),
    (([4, 0, 4, 3, 3], 5), 2.8),
    (([7, 7, 1, 1, 9, 9], 2), 9.0),
])

<details><summary><b>Hint 1</b></summary>

Signal: a block of fixed length that moves one step at a time. Pattern: **fixed-size sliding window**.

</details>

<details><summary><b>Hint 2</b></summary>

1. Sum the first `k` values: `window`.
2. For `i` from `k` to the end: `window += nums[i] - nums[i - k]` (one value enters, one leaves).
3. Track the best sum; return `best / k` (divide once at the end).

</details>

<details><summary><b>Solution</b></summary>

```python
def find_max_average(nums, k):
    window = sum(nums[:k])
    best = window
    for i in range(k, len(nums)):
        window += nums[i] - nums[i - k]
        best = max(best, window)
    return best / k
```

**Why:** Neighbouring windows share k-1 values, so you only add the new value and remove the old one. That is O(1) work per step instead of re-summing k values. Comparing sums is the same as comparing averages because k is fixed.

**Complexity:** O(n) time, O(1) space.

**Common mistakes:** Starting `best` at 0 (wrong when every value is negative, as in `[-1, -2, -3]`). Re-summing each window with `sum(nums[i:i+k])` (O(n * k)). Integer division `//`.

**Learn more:** [algo-sliding-window](https://mahsa-agz.github.io/ds-handbook/#algo-sliding-window)

</details>

---
### Q3. Mirror the tree

*easy, about 10 min*

Turn a binary tree into its mirror image: for every node, the left and right children swap places.
Return the root.

Examples (level order):
- `[4, 2, 7, 1, 3, 6, 9]` -> `[4, 7, 2, 9, 6, 3, 1]`
- `[2, 1, 3]` -> `[2, 3, 1]`
- `[1, 2]` -> `[1, None, 2]`

*Source: [LeetCode 226](https://leetcode.com/problems/invert-binary-tree/)*

In [ ]:
def invert_tree(root):
    pass

# tests
def mirrored(vals):
    return tree_values(invert_tree(build_tree(vals)))

check(mirrored, [
    ([4, 2, 7, 1, 3, 6, 9], [4, 7, 2, 9, 6, 3, 1]),
    ([2, 1, 3], [2, 3, 1]),
    ([], []),
    ([1], [1]),
    ([1, 2], [1, None, 2]),
    ([1, 2, None, 3], [1, None, 2, None, 3]),
])

<details><summary><b>Hint 1</b></summary>

Signal: do the same small action at every node of the tree. Pattern: **tree traversal (DFS or BFS)**.

</details>

<details><summary><b>Hint 2</b></summary>

1. Base case: `None` -> return `None`.
2. Swap: `root.left, root.right = root.right, root.left`.
3. Recurse into both children (or push them on a stack / queue). Return `root`.

</details>

<details><summary><b>Solution</b></summary>

```python
def invert_tree(root):
    if root is None:
        return None
    root.left, root.right = invert_tree(root.right), invert_tree(root.left)
    return root

# iterative version (no recursion limit):
def invert_tree_iter(root):
    stack = [root]
    while stack:
        node = stack.pop()
        if node:
            node.left, node.right = node.right, node.left
            stack += [node.left, node.right]
    return root

print(tree_values(invert_tree_iter(build_tree([2, 1, 3]))))  # [2, 3, 1]
```

**Why:** A mirror image is just "swap the children" applied at every node; the order of visiting does not matter, so any traversal works. Each node is touched once.

**Complexity:** O(n) time; O(h) space for recursion or the stack.

**Common mistakes:** Swapping only the root's children. Writing `root.left = invert_tree(root.right)` and then `root.right = invert_tree(root.left)` on two lines: the second line reads the already changed left child. Use one tuple assignment or a temp variable.

**Learn more:** [algo-trees](https://mahsa-agz.github.io/ds-handbook/#algo-trees)

</details>

---
### Q4. Squeeze out repeats in a sorted list (bonus)

*easy, about 10 min*

Optional. `nums` is sorted. Rearrange it in place so that its first `k` positions hold each distinct
value once, in order, and return `k`. What is left after position `k` does not matter.

Examples:
- `[1, 1, 2]` -> `k = 2`, first two values `[1, 2]`
- `[0, 0, 1, 1, 1, 2, 2, 3, 3, 4]` -> `k = 5`, first five values `[0, 1, 2, 3, 4]`

*Source: [LeetCode 26](https://leetcode.com/problems/remove-duplicates-from-sorted-array/)*

In [ ]:
def remove_duplicates(nums):
    pass

# tests
def dedup(nums):
    k = remove_duplicates(nums)
    return (k, nums[:k]) if isinstance(k, int) else k

check(dedup, [
    ([1, 1, 2], (2, [1, 2])),
    ([0, 0, 1, 1, 1, 2, 2, 3, 3, 4], (5, [0, 1, 2, 3, 4])),
    ([7], (1, [7])),
    ([2, 2, 2], (1, [2])),
    ([-3, -1, 0, 5], (4, [-3, -1, 0, 5])),
])

<details><summary><b>Hint 1</b></summary>

Signal: sorted input, in-place rewrite, keep order. Pattern: **two pointers** (slow write, fast read).

</details>

<details><summary><b>Hint 2</b></summary>

1. `k = 1` (the first value is always kept; handle an empty list separately).
2. For `i` from 1: if `nums[i] != nums[k - 1]`, write `nums[k] = nums[i]` and `k += 1`.
3. Return `k`.

</details>

<details><summary><b>Solution</b></summary>

```python
def remove_duplicates(nums):
    if not nums:
        return 0
    k = 1
    for i in range(1, len(nums)):
        if nums[i] != nums[k - 1]:
            nums[k] = nums[i]
            k += 1
    return k
```

**Why:** Because the list is sorted, equal values sit next to each other, so a value is new exactly when it differs from the last kept value. The write pointer never passes the read pointer, so nothing is overwritten before it is read.

**Complexity:** O(n) time, O(1) space.

**Common mistakes:** Using `set(nums)` (loses the in-place requirement and the order guarantee). Deleting from the list while looping over it.

**Learn more:** [algo-two-pointers](https://mahsa-agz.github.io/ds-handbook/#algo-two-pointers)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [SQL](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_05/2_sql.ipynb)